In [1]:
from groq import Groq
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

# Chat API : GROQ

In [2]:
load_dotenv()
client = Groq(api_key=os.getenv('GROQ_API_KEY'))
llm_model = "llama-3.3-70b-versatile"

In [3]:
def get_completion(propmt, model=llm_model):

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": propmt}],
        temperature=0
    )

    return response.choices[0].message.content

In [4]:
get_completion("what is 1 + 1")

'1 + 1 = 2.'

In [5]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [6]:
style = """American English \
in a calm and respectful tone
"""

In [7]:
prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [8]:
response = get_completion(prompt)

In [9]:
response

"I'm having a bit of a frustrating issue with my blender. The lid came off unexpectedly and splattered my kitchen walls with smoothie. To make matters worse, the warranty doesn't cover the cost of cleaning up the mess. I would greatly appreciate your assistance with this problem as soon as possible."

# Chat API : LangChain

### Model

In [10]:
chat = ChatGroq(model=llm_model, temperature=0.0)
chat

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x76b67e5a0690>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x76b67e58d2b0>, model_name='llama-3.3-70b-versatile', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### Prompt Template

In [11]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [12]:
prompt_template = ChatPromptTemplate.from_template(template_string)

In [13]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

### Customer Email

In [14]:
customer_style = """American English \
in a calm and respectful tone
"""

In [15]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [16]:
customer_messages = prompt_template.format_messages(
    style=customer_style,
    text=customer_email
)

In [17]:
type(customer_messages), type(customer_messages[0].content)

(list, str)

In [18]:
customer_response = chat.invoke(customer_messages)

In [19]:
print(customer_response.content)

I'm having a bit of a frustrating moment. My blender lid came loose and splattered smoothie all over my kitchen walls. To make things even more challenging, the warranty doesn't cover the cost of cleaning up the mess. I would greatly appreciate your assistance with this issue as soon as possible.


### Service Reply

In [20]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

In [21]:
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

In [22]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply
)

In [23]:
service_messages[0].content

"Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!\n```\n"

In [24]:
service_response = chat.invoke(service_messages)

In [25]:
service_response.content

"```Ahoy, valued customer, I be needin' to inform ye that the warranty don't exactly cover the cleanin' expenses for yer kitchen, savvy? It seems that the mishap occurred due to a wee bit o' misuse o' yer trusty blender, specifically forgettin' to secure the lid before setlin' her to work. Now, I know it be a right proper nuisance, but I be afraid it falls under the category o' user error, matey. So hoist the colors and take a deep breath, for it seems ye be responsible for the mess. Mayhap next time, ye'll be more mindful o' the lid, aye? Fair winds, and may yer future blendin' adventures be spill-free!```"

## Output Parser

In [26]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [27]:
prompt_template = ChatPromptTemplate.from_template(review_template)
prompt_template

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})])

In [28]:
messages = prompt_template.format_messages(
    text=customer_review
)

In [29]:
messages[0].content

"For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.\n\n"

In [30]:
response = chat.invoke(messages)

In [31]:
## so this is actually not a Json it is a string

type(response.content)

str

## Parse the LLM output string into a Python dictionary

In [38]:
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser

In [42]:
gift_schema = ResponseSchema(name="gift",
                             description="Was the item purchased\
                             as a gift for someone else? \
                             Answer True if yes,\
                             False if not or unknown.")
delivery_days_schema = ResponseSchema(name="delivery_days",
                                      description="How many days\
                                      did it take for the product\
                                      to arrive? If this \
                                      information is not found,\
                                      output -1.")
price_value_schema = ResponseSchema(name="price_value",
                                    description="Extract any\
                                    sentences about the value or \
                                    price, and output them as a \
                                    comma separated Python list.")

response_schemas = [gift_schema, 
                    delivery_days_schema,
                    price_value_schema]

In [43]:
output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

In [44]:
output_parser

StructuredOutputParser(response_schemas=[ResponseSchema(name='gift', description='Was the item purchased                             as a gift for someone else?                              Answer True if yes,                             False if not or unknown.', type='string'), ResponseSchema(name='delivery_days', description='How many days                                      did it take for the product                                      to arrive? If this                                       information is not found,                                      output -1.', type='string'), ResponseSchema(name='price_value', description='Extract any                                    sentences about the value or                                     price, and output them as a                                     comma separated Python list.', type='string')])

In [45]:
format_instructions = output_parser.get_format_instructions()

In [47]:
format_instructions

'The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":\n\n```json\n{\n\t"gift": string  // Was the item purchased                             as a gift for someone else?                              Answer True if yes,                             False if not or unknown.\n\t"delivery_days": string  // How many days                                      did it take for the product                                      to arrive? If this                                       information is not found,                                      output -1.\n\t"price_value": string  // Extract any                                    sentences about the value or                                     price, and output them as a                                     comma separated Python list.\n}\n```'

In [48]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

{format_instructions}
"""

In [50]:
prompt_template = ChatPromptTemplate.from_template(template=review_template_2)
prompt_template

ChatPromptTemplate(input_variables=['format_instructions', 'text'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['format_instructions', 'text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the productto arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\ntext: {text}\n\n{format_instructions}\n'), additional_kwargs={})])

In [51]:
messages = prompt_template.format_messages(
    text=customer_review,
    format_instructions=format_instructions
)

In [53]:
messages[0].content

'For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the productto arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\ntext: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife\'s anniversary present. I think my wife liked it so much she was speechless. So far I\'ve been the only one using it, and I\'ve been using it every other morning to clear the leaves on our lawn. It\'s slightly more expensive than the other leaf blowers out there, but I think it\'s worth it for the extra features.\n\n\nThe output should be a markdown code snippet formatted in the following schema, including the leading and trailing "

In [54]:
response = chat.invoke(messages)

In [58]:
output_dic = output_parser.parse(response.content)

In [61]:
type(output_dic)

dict

In [64]:
output_dic.get("gift"), output_dic

('True',
 {'gift': 'True',
  'delivery_days': '2',
  'price_value': "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."})